<a href="https://colab.research.google.com/github/csrsustain/DEC-questionnaire/blob/main/AHU_Energy_and_Cost_Calculation_(Mixed_Air_Dampers).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
import pandas as pd

# ── 1. ASSUMPTIONS ───────────────────────────────────
AIR_DENSITY       = 1.27      # kg/m³
SPECIFIC_HEAT     = 1.007     # kJ/kg/K
LPHW_EFFICIENCY   = 0.90     # boiler/LPHW efficiency
GAS_COST          = 0.05      # £/kWh (5p)

# ── 2. SYSTEM OPERATION & EQUIPMENT ─────────────────────────────
# Damper Configurations
# Current State (e.g., 100% Fresh Air as a typical baseline)
CURRENT_RECIRC_DAMPER = 0.5 # 0% Recirc
CURRENT_FRESH_DAMPER  = 0.5 # 100% Fresh air

# Desired State (e.g., optimized mixed air)
DESIRED_RECIRC_DAMPER = 1.0 # 80% Return air
DESIRED_FRESH_DAMPER  = 0.10 # 20% Fresh air

RETURN_TEMP   = 19.9  # Constant return air temperature from space (°C)

# AHU Operating Schedule (manual input)
AHU_OPERATING_START_HOUR = 6  # 6 AM (inclusive) This means that, in the first step, the engine filters the weather data to the specified time frame (7AM - 7PM)
AHU_OPERATING_END_HOUR   = 18 # 6 PM (exclusive, so up to 18:59)

OPERATING_HOURS_PER_DAY = 12 # Hours the AHU operates each day it's on
OPERATING_DAYS_PER_WEEK = 5  # Days per week the AHU operates
OPERATING_WEEKS_PER_YEAR = 52 # Weeks per year the AHU operates
# By summing these variables, we calculate the annual operating hours, which are then divided by the total number of hours in a year (8,760)

# Calculate ANNUAL_OPERATING_HOURS from the granular inputs
ANNUAL_OPERATING_HOURS = OPERATING_HOURS_PER_DAY * OPERATING_DAYS_PER_WEEK * OPERATING_WEEKS_PER_YEAR  #This is the total AHU operating hours

# Quick validation check to ensure invalid inputs are excluded from the calculation
if round(CURRENT_RECIRC_DAMPER + CURRENT_FRESH_DAMPER, 6) != 1.0:
    raise ValueError(f"Current Damper positions must add up to 1.0 (got {CURRENT_RECIRC_DAMPER + CURRENT_FRESH_DAMPER})")
if round(DESIRED_RECIRC_DAMPER + DESIRED_FRESH_DAMPER, 6) != 1.1:
    raise ValueError(f"Desired Damper positions must add up to 1.1 (got {DESIRED_RECIRC_DAMPER + DESIRED_FRESH_DAMPER})")
if AHU_OPERATING_START_HOUR >= AHU_OPERATING_END_HOUR:
    raise ValueError("AHU_OPERATING_START_HOUR must be less than AHU_OPERATING_END_HOUR.")
if not (0 <= AHU_OPERATING_START_HOUR <= 23 and 0 <= AHU_OPERATING_END_HOUR <= 24):
    raise ValueError("AHU_OPERATING_START_HOUR and AHU_OPERATING_END_HOUR must be valid hour values (0-24).") # This ensures that only valid numbers between 01 and 24 can be entered
if not (0 <= OPERATING_HOURS_PER_DAY <= 24):
    raise ValueError("OPERATING_HOURS_PER_DAY must be between 0 and 24.")
if not (0 <= OPERATING_DAYS_PER_WEEK <= 7):
    raise ValueError("OPERATING_DAYS_PER_WEEK must be between 0 and 7.")
if not (0 <= OPERATING_WEEKS_PER_YEAR <= 52):
    raise ValueError("OPERATING_WEEKS_PER_YEAR must be between 0 and 52.")

equipment = [
    {"name": "AHU - Sample", "air_flow_m3s": 15, "temp_setpoint": 21, "hours_per_year": ANNUAL_OPERATING_HOURS},
]

# ── 3. LOAD WEATHER DATA ─────────────────────────────
WEATHER_FILE = "Manchester hourly weather data-2025.xlsx"

# Read the Excel file with the correct sheet name and header row
weather_df = pd.read_excel(WEATHER_FILE, sheet_name="0 hourly", header=0)

# Select the relevant columns and rename them for consistency
weather_df = weather_df[["Timestamp (UK/Manchester)", "Weather, Temperature, ºC"]].dropna()
weather_df.columns = ["timestamp", "outdoor_temp_c"]

weather_df["datetime"] = pd.to_datetime(weather_df["timestamp"])
weather_df["month"] = weather_df["datetime"].dt.strftime("%b")

# Filter weather data for operating hours (Because the building is occupied between 7:00 AM and 7:00 PM)
weather_df["hour"] = weather_df["datetime"].dt.hour
weather_df = weather_df[(weather_df["hour"] >= AHU_OPERATING_START_HOUR) & (weather_df["hour"] < AHU_OPERATING_END_HOUR)]

# Filter weather data for operating days of the week (0=Mon ... 6=Sun).
# Without this, the dataset still contains weekend hours, and the run_fraction
# multiplier below ends up double-discounting instead of properly excluding them.
weather_df["weekday"] = weather_df["datetime"].dt.weekday
weather_df = weather_df[weather_df["weekday"] < OPERATING_DAYS_PER_WEEK]

# ── 4. ENERGY CALCULATION ENGINE ─────────────────────────────
def calculate_thermal_demand(outdoor_temp, air_flow, temp_setpoint, recirc_damper, fresh_damper):
    """
    Calculates the thermal heating demand for a given damper configuration.
    """
    if fresh_damper == 1.0: # 100% Fresh Air scenario
        delta_t = temp_setpoint - outdoor_temp
    else:             # Mixed air scenario
        mixed_air_temp = (recirc_damper * RETURN_TEMP) + (fresh_damper * outdoor_temp)
        delta_t = temp_setpoint - mixed_air_temp

    thermal_kw = air_flow * AIR_DENSITY * SPECIFIC_HEAT * abs(delta_t)

    if delta_t > 0:  # Heating needed
        heating_kw = thermal_kw / LPHW_EFFICIENCY
    else:             # Cooling needed (ignored/handled locally for heating calculations)
        heating_kw = 0
    return heating_kw


# ── 5. RUN PERFORMANCE & FINANCIAL ANALYSIS ─────────────────
HOURS_IN_YEAR = 8760
INTERVAL_HRS  = 1.0   # hourly data

print("=" * 75)
print("             AHU RECIRCULATION ENERGY & COST SAVINGS REPORT")
print("=" * 75)
print(f" Current Damper Configuration: {CURRENT_RECIRC_DAMPER*100:.0f}% Recirc / {CURRENT_FRESH_DAMPER*100:.0f}% Fresh")
print(f" Desired Damper Configuration: {DESIRED_RECIRC_DAMPER*100:.0f}% Recirc / {DESIRED_FRESH_DAMPER*100:.0f}% Fresh")
print(f" Design Conditions   : Return Air Temp: {RETURN_TEMP}°C | Gas Cost: £{GAS_COST:.2f}/kWh")
print(f" AHU Operating Hours (Daily): {AHU_OPERATING_START_HOUR}:00 - {AHU_OPERATING_END_HOUR}:00")
print(f" AHU Operating Schedule: {OPERATING_HOURS_PER_DAY} hrs/day, {OPERATING_DAYS_PER_WEEK} days/week, {OPERATING_WEEKS_PER_YEAR} weeks/year")
print(f" Calculated Annual Operating Hours: {ANNUAL_OPERATING_HOURS} hours")
print("=" * 75)

for unit in equipment:
    # Weather data is now already filtered to the exact operating hours AND
    # operating weekdays, so each row IS a real operating hour — no extra
    # run_fraction scaling is needed (and applying one would double-discount).
    current_state_heat_kw = weather_df.apply(
        lambda row: calculate_thermal_demand(row["outdoor_temp_c"], unit["air_flow_m3s"], unit["temp_setpoint"], CURRENT_RECIRC_DAMPER, CURRENT_FRESH_DAMPER),
        axis=1
    )

    # Calculate heating for Desired State
    desired_state_heat_kw = weather_df.apply(
        lambda row: calculate_thermal_demand(row["outdoor_temp_c"], unit["air_flow_m3s"], unit["temp_setpoint"], DESIRED_RECIRC_DAMPER, DESIRED_FRESH_DAMPER),
        axis=1
    )

    # Create a results DataFrame for easier processing
    results = pd.DataFrame({
        "current_state_heat_kw": current_state_heat_kw,
        "desired_state_heat_kw": desired_state_heat_kw
    })

    # Convert kW demands to kWh consumed within the time interval
    results["current_state_kwh"] = results["current_state_heat_kw"] * INTERVAL_HRS
    results["desired_state_kwh"]    = results["desired_state_heat_kw"] * INTERVAL_HRS

    # Savings per interval are now a direct net calculation of the two columns
    results["saved_heat_kwh"]    = results["current_state_kwh"] - results["desired_state_kwh"]

    # Sum up annual metrics
    total_current_state_kwh = results["current_state_kwh"].sum()
    total_desired_state_kwh    = results["desired_state_kwh"].sum()
    total_saved_kwh    = results["saved_heat_kwh"].sum()

    current_state_cost = total_current_state_kwh * GAS_COST
    desired_state_cost    = total_desired_state_kwh * GAS_COST
    total_savings = current_state_cost - desired_state_cost

    print(f"\nEquipment : {unit['name']}")
    print(f"Air Flow  : {unit['air_flow_m3s']} m³/s  |  Setpoint: {unit['temp_setpoint']}°C  |  Run Hours: {unit['hours_per_year']}/yr")
    print("-" * 75)

    # Attach results to dataframe for monthly aggregation
    weather_df_copy = weather_df.copy()
    weather_df_copy["current_state_kwh"] = results["current_state_kwh"].values
    weather_df_copy["desired_state_kwh"]    = results["desired_state_kwh"].values
    weather_df_copy["saved_kwh"]    = results["saved_heat_kwh"].values

    month_order = ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"]
    monthly = weather_df_copy.groupby("month")[["current_state_kwh","desired_state_kwh","saved_kwh"]].sum()
    monthly = monthly.reindex(month_order).fillna(0)

    # Print out monthly table
    print(f"{'Month':<6} {'Current State (kWh)':>20} {'Desired State (kWh)':>20} {'Energy Saved (kWh)':>19} {'Cost Saved (£)':>14}")
    print("-" * 75)
    for month, row in monthly.iterrows():
        m_savings_pounds = row["saved_kwh"] * GAS_COST
        print(f"{month:<6} {row['current_state_kwh']:>20,.0f} {row['desired_state_kwh']:>20,.0f} {row['saved_kwh']:>19,.0f} {m_savings_pounds:>14,.2f}")

    print("-" * 75)
    print(f"{'TOTAL':<6} {total_current_state_kwh:>20,.0f} {total_desired_state_kwh:>20,.0f} {total_saved_kwh:>19,.0f}  £{total_savings:>12,.2f}")
    print("-" * 75)
    print(f"  ► Annual Cost if Current State:  £{current_state_cost:,.2f}")
    print(f"  ► Annual Cost with Desired State   :  £{desired_state_cost:,.2f}")
    print(f"  ♣ TOTAL NET ANNUAL SAVINGS     :  £{total_savings:,.2f}")
    print("=" * 75)

             AHU RECIRCULATION ENERGY & COST SAVINGS REPORT
 Current Damper Configuration: 50% Recirc / 50% Fresh
 Desired Damper Configuration: 100% Recirc / 10% Fresh
 Design Conditions   : Return Air Temp: 19.9°C | Gas Cost: £0.05/kWh
 AHU Operating Hours (Daily): 6:00 - 18:00
 AHU Operating Schedule: 12 hrs/day, 5 days/week, 52 weeks/year
 Calculated Annual Operating Hours: 3120 hours

Equipment : AHU - Sample
Air Flow  : 15 m³/s  |  Setpoint: 21°C  |  Run Hours: 3120/yr
---------------------------------------------------------------------------
Month   Current State (kWh)  Desired State (kWh)  Energy Saved (kWh) Cost Saved (£)
---------------------------------------------------------------------------
Jan                 103,928                8,087              95,841       4,792.04
Feb                  78,716                4,839              73,877       3,693.85
Mar                  65,305                2,598              62,708       3,135.38
Apr                  41,504     